In [ ]:
import sys, os
os.chdir(os.path.abspath('..'))
sys.path.insert(0, os.getcwd())

### cids that are in Nymph family with genera on Lepid tree but not on Nymph tree (this was previously used for a pairwise self-similarity test)

In [ ]:
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph


tree_lepid = build_tree_lepid()
tree_nymph = build_tree_nymph()

cids_lepid = [tip.name for tip in tree_lepid.get_terminals()]
cids_nymph = [tip.name for tip in tree_nymph.get_terminals()]

class_data_lepid = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")

genera_lepid = set([cid.split("_")[0] for cid in cids_lepid])
genera_nymph = set([cid.split("_")[0] for cid in cids_nymph])
genera_lepid_non_nymph = sorted(genera_lepid - genera_nymph)

cids_genera_lepid_non_nymph_nymphalidae = set()
for cid in class_data_lepid.keys():
    if class_data_lepid[cid]["family"] == "nymphalidae":
        if cid.split("_")[0] in genera_lepid_non_nymph:
            cids_genera_lepid_non_nymph_nymphalidae.add(cid)

In [ ]:
[cid for cid in cids_lepid if cid in cids_genera_lepid_non_nymph_nymphalidae]

In [ ]:
from itertools import combinations

from utils.phylo import get_tree
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import combine_trees_lepid_nymph
from preprocessing.common.phylo import prune_tree, augment_class_data

In [ ]:
def print_pairwise_distances(tree, labels):
    for a, b in combinations(labels, 2):
        print(f"tree.distance('{a}', '{b}') = {tree.distance(a, b)}")

def cids_starting_with(cids, prefix):
    return sorted([cid for cid in cids if cid.startswith(prefix)])

def labels_in_cids(cids, labels, header=None):
    if header:
        print(header)
    for cid in labels:
        print(cid in cids)

def cids_in_rank(cid, rank, class_data):
    rank_name = class_data[cid][rank]
    return sorted([cid for cid in class_data.keys() if class_data[cid][rank] == rank_name])

def neighbor_genus_dists(cid, tree, class_data, remove_genus_cid=True):

    cids_tree = {tip.name for tip in tree.get_terminals()}

    cd_cid = class_data[cid]
    rank = "family"
    if cd_cid["subfamily"] is not None:
        rank = "subfamily"
    if cd_cid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    cids_cands = set(cids_in_rank(cid, rank, class_data)) & cids_tree
    if remove_genus_cid:
        cids_cands = {cid for cid in cids_cands if class_data[cid]["genus"] != cd_cid["genus"]}

    for cid_cand in sorted(cids_cands):
        print(f"{cid_cand}: {tree.distance(cid, cid_cand)}")

def closest_species_outside_genus(cid, tree, class_data):

    cids_tree = {tip.name for tip in tree.get_terminals()}

    cd_cid = class_data[cid]
    rank = "family"
    if cd_cid["subfamily"] is not None:
        rank = "subfamily"
    if cd_cid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    cids_cands = set(cids_in_rank(cid, rank, class_data)) & cids_tree
    cids_cands = {cid for cid in cids_cands if class_data[cid]["genus"] != cd_cid["genus"]}

    cid_closest = None
    dist_closest = float("inf")
    for cid_cand in sorted(cids_cands):
        dist = tree.distance(cid, cid_cand)
        if dist < dist_closest:
            cid_closest = cid_cand
            dist_closest = dist

    print(f"Closest species outside genus: {cid_closest} (distance {dist_closest:.2f})")

In [ ]:
tree = get_tree(dataset="lepid")  # merged
cids_tree = {tip.name for tip in tree.get_terminals()}

tree_lepid = build_tree_lepid()
tree_nymph = build_tree_nymph()
cids_lepid = {tip.name for tip in tree_lepid.get_terminals()}
cids_nymph = {tip.name for tip in tree_nymph.get_terminals()}

class_data = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")
class_data_aug = augment_class_data(class_data, tree)

### closest genus to "colias" genus: "zerene"

In [ ]:
neighbor_genus_dists('colias_palaeno', tree, class_data_aug, remove_genus_cid=False)

### cid-combo distances

In [ ]:
labels = ['adelpha_mesentina', 'aglais_urticae', 'aglais_io', 'daedalma_dinias']
print_pairwise_distances(tree, labels)